# Markowitz Portfolios under Transaction Costs

**Authors:** Olivier Ledoit, Michael Wolf

**Published:** 2025-01-13

**URL:** [https://doi.org/10.1016/j.qref.2025.101962](https://doi.org/10.1016/j.qref.2025.101962)

**Abstract:**
Markowitz portfolio selection is a cornerstone in finance, in academia as well as in the industry. Most academic studies either ignore transaction costs or account for them in a way that is both unrealistic and suboptimal by (i) assuming transaction costs to be constant across stocks and (ii) ignoring them at the portfolio-selection state and simply paying them after the fact. Our paper proposes a method to fix both shortcomings. As we show, if transaction costs are accounted for (properly) at the portfolio-selection stage, net performance in terms of the Sharpe ratio often increases, in particular for high-turnover strategies.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1: Configuration

In [ ]:
# Configuration
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Set display options
pd.set_option('display.float_format', lambda x: '%.5f' % x)

# Define tickers and date range
tickers = ['AAPL', 'MSFT', 'GOOGL']
start_date = '2020-01-01'
end_date = '2023-01-01'

## Phase 2: Data Download and Feature Engineering

In [ ]:
# Download data
data = yf.download(tickers, start=start_date, end=end_date)['Adj Close']

# Calculate daily returns
returns = data.pct_change().dropna()

## Phase 3: Signal Generation and Portfolio Construction

In [ ]:
# Simple moving average strategy
short_window = 40
long_window = 100

signals = pd.DataFrame(index=returns.index)
signals['signal'] = 0
signals['short_mavg'] = returns.rolling(window=short_window, min_periods=1).mean()
signals['long_mavg'] = returns.rolling(window=long_window, min_periods=1).mean()

# Generate signals
signals['signal'][short_window:] = np.where(signals['short_mavg'][short_window:] 
                                             > signals['long_mavg'][short_window:], 1, 0)
signals['positions'] = signals['signal'].diff()

## Phase 4: Vectorized Backtest

In [ ]:
# Shift signals by 1 to avoid look-ahead bias
signals['signal'] = signals['signal'].shift(1)

# Calculate strategy returns
strategy_returns = signals['signal'] * returns
cumulative_strategy_returns = (1 + strategy_returns).cumprod()

## Phase 5: Performance Metrics

In [ ]:
# Calculate performance metrics
def calculate_performance(returns):
    sharpe_ratio = np.sqrt(252) * returns.mean() / returns.std()
    downside_returns = returns[returns < 0]
    sortino_ratio = np.sqrt(252) * returns.mean() / downside_returns.std()
    max_drawdown = (cumulative_strategy_returns / cumulative_strategy_returns.cummax() - 1).min()
    calmar_ratio = returns.mean() * 252 / abs(max_drawdown)
    return sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

sharpe, sortino, calmar, max_dd = calculate_performance(strategy_returns)

# Plot equity curve
plt.figure(figsize=(10, 6))
plt.plot(cumulative_strategy_returns, label='Strategy')
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

# Display performance metrics
print(f"Sharpe Ratio: {sharpe:.2f}")
print(f"Sortino Ratio: {sortino:.2f}")
print(f"Calmar Ratio: {calmar:.2f}")
print(f"Max Drawdown: {max_dd:.2f}")

## Phase 6: Monitoring Stub

In [ ]:
# Monitoring stub - to be implemented
# This could include real-time data fetching, alerting, etc.

print("Monitoring system is not implemented in this notebook.")